In [ ]:
# Path: /mnt/data/clean_image_script.py

import cv2
import hashlib
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from PIL import Image

# Configuration
IMAGE_DIRECTORY = "Dataset"
OUTPUT_CSV = "dataset_cleaned.csv"
HASH_ALGORITHM = "md5"
CHUNK_SIZE = 4096
IMAGE_EXTENSION = '.jpg'

# Initialize lists for storing image dimensions
widths, heights = [], []

# Calculate the hash of an image file using the specified algorithm
def calculate_image_hash(image_path, algorithm=HASH_ALGORITHM, chunk_size=CHUNK_SIZE):
    hash_object = hashlib.new(algorithm)
    with open(image_path, "rb") as file:
        while chunk := file.read(chunk_size):
            hash_object.update(chunk)
    return hash_object.hexdigest()

# Load and clean dataset
def load_and_clean_dataset(directory, extension):
    image_data, hashes = [], {}
    for root, _, files in os.walk(directory):
        label = os.path.basename(root)
        for file_name in files:
            if file_name.lower().endswith(extension):
                file_path = os.path.join(root, file_name)
                file_hash = calculate_image_hash(file_path)

                if file_hash not in hashes:
                    image_data.append({'filepath': file_path, 'label': label})
                    hashes[file_hash] = file_path
                else:
                    print(f"Duplicate detected: {file_path}")

    df = pd.DataFrame(image_data)
    df.to_csv(OUTPUT_CSV, index=False)
    return df

# Load previously saved cleaned dataset or run the cleaning process
try:
    df = pd.read_csv(OUTPUT_CSV)
except FileNotFoundError:
    df = load_and_clean_dataset(IMAGE_DIRECTORY, IMAGE_EXTENSION)


In [ ]:
# Visualize sample images per class

NUM_SAMPLES_PER_CLASS = 5

def show_samples_per_class(data_frame, num_samples):
    """Display sample images per class using matplotlib."""
    classes = data_frame['label'].unique()
    for cls in classes:
        cls_df = data_frame[data_frame['label'] == cls].sample(min(num_samples, len(data_frame[data_frame['label'] == cls])))
        fig, axs = plt.subplots(1, len(cls_df), figsize=(15, 15))
        fig.suptitle(f'Samples from Class: {cls}', fontsize=16)
        for i, (_, row) in enumerate(cls_df.iterrows()):
            img = Image.open(row['filepath'])
            axs[i].imshow(img)
            axs[i].axis('off')
        plt.show()

show_samples_per_class(df, NUM_SAMPLES_PER_CLASS)


In [ ]:
# Analyze and visualize image dimensions and aspect ratios

# Initialize lists for image dimensions and aspect ratios
aspect_ratios = []

# Analyze image dimensions and aspect ratios
for _, row in df.iterrows():
    img = Image.open(row['filepath'])
    width, height = img.size
    widths.append(width)
    heights.append(height)
    aspect_ratios.append(width / height)

# Plot histograms for image dimensions and aspect ratios
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

sns.histplot(widths, bins=30, kde=True, ax=axs[0], color='blue')
axs[0].set_title('Image Width Distribution')
axs[0].set_xlabel('Width (pixels)')

sns.histplot(heights, bins=30, kde=True, ax=axs[1], color='orange')
axs[1].set_title('Image Height Distribution')
axs[1].set_xlabel('Height (pixels)')

sns.histplot(aspect_ratios, bins=30, kde=True, ax=axs[2], color='green')
axs[2].set_title('Aspect Ratio Distribution')
axs[2].set_xlabel('Aspect Ratio (Width/Height)')

plt.show()


In [ ]:
# Importing Required Libraries
import numpy as np
import pandas as pd
import cv2
import os
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import tensorflow as tf

# Step 1: Data Loading and Preprocessing
def load_data(data_dir):
    images = []
    labels = []
    for label_dir in os.listdir(data_dir):
        label_path = os.path.join(data_dir, label_dir)
        for img_name in os.listdir(label_path):
            img_path = os.path.join(label_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            images.append(img)
            labels.append(label_dir)
    return np.array(images), np.array(labels)

data_dir = '/path/to/dataset'
images, labels = load_data(data_dir)

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Step 2: Model Selection and Evaluation
def build_model():
    vgg = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    for layer in vgg.layers:
        layer.trainable = False

    x = vgg.output
    x = Flatten()(x)
    x = Dense(4096, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(4096, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=vgg.input, outputs=x)
    return model

model = build_model()
model.compile(optimizer=Adam(lr=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# Step 3: Model Training and Optimization
train_images, val_images, train_labels, val_labels = train_test_split(images, labels, test_size=0.2, random_state=42)

train_generator = datagen.flow(train_images, train_labels, batch_size=32)
val_generator = datagen.flow(val_images, val_labels, batch_size=32)

history = model.fit(
    train_generator,
    steps_per_epoch=len(train_images) // 32,
    validation_data=val_generator,
    validation_steps=len(val_images) // 32,
    epochs=25
)

# Step 4: Evaluating the Model
def plot_history(history):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

plot_history(history)

# Step 5: Saving the Model
model.save('face_verification_model.h5')

# Step 6: Inference Function for Face Verification
def verify_face(model, img_path, threshold=0.5):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    prediction = model.predict(img_array)[0][0]
    return prediction > threshold

# Test the function
test_image_path = '/path/to/test/image.jpg'
is_verified = verify_face(model, test_image_path)
print('Face Verified' if is_verified else 'Face Not Verified')

# Step 7: Liveness Detection (Simple Implementation)
def detect_liveness(img_path):
    # Placeholder for liveness detection implementation
    # This can include techniques like blinking detection, texture analysis, etc.
    return True  # Assuming liveness detected for this example

# Test liveness detection
liveness = detect_liveness(test_image_path)
print('Liveness Detected' if liveness else 'Spoof Attempt')

# Step 8: Ensuring Privacy (Sample Encryption of Image Data)
from cryptography.fernet import Fernet

def encrypt_image(img_path, key):
    with open(img_path, 'rb') as file:
        image_data = file.read()
    fernet = Fernet(key)
    encrypted = fernet.encrypt(image_data)
    return encrypted

def decrypt_image(encrypted_data, key):
    fernet = Fernet(key)
    decrypted = fernet.decrypt(encrypted_data)
    return decrypted

key = Fernet.generate_key()
encrypted_image = encrypt_image(test_image_path, key)
decrypted_image = decrypt_image(encrypted_image, key)
print('Image Encryption and Decryption Successful')

# Step 9: Implement Biometric Reading for Access (Sample Implementation)
def biometric_reading_for_access(model, img_path):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            print("Access Granted")
        else:
            print("Spoof Attempt Detected, Access Denied")
    else:
        print("Face Not Verified, Access Denied")

# Test Biometric Reading for Access
biometric_reading_for_access(model, test_image_path)

# Step 10: Digital Payments Security (Sample Implementation)
def secure_digital_payment(model, img_path):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            print("Payment Authorized")
        else:
            print("Spoof Attempt Detected, Payment Denied")
    else:
        print("Face Not Verified, Payment Denied")

# Test Digital Payments Security
secure_digital_payment(model, test_image_path)

# Step 11: Attendance Tracking (Sample Implementation)
def track_attendance(model, img_path, attendance_db):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            print("Attendance Marked")
            attendance_db.append(img_path)
        else:
            print("Spoof Attempt Detected, Attendance Not Marked")
    else:
        print("Face Not Verified, Attendance Not Marked")

# Test Attendance Tracking
attendance_db = []
track_attendance(model, test_image_path, attendance_db)
print(attendance_db)

# Step 12: Employee Time and Attendance Tracking (Sample Implementation)
def track_employee_time_and_attendance(model, img_path, employee_db):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            print("Employee Check-In Marked")
            employee_db.append(img_path)
        else:
            print("Spoof Attempt Detected, Check-In Not Marked")
    else:
        print("Face Not Verified, Check-In Not Marked")

# Test Employee Time and Attendance Tracking
employee_db = []
track_employee_time_and_attendance(model, test_image_path, employee_db)
print(employee_db)

# Finished Implementation


In [ ]:
# Importing Required Libraries
import numpy as np
import pandas as pd
import cv2
import os
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import tensorflow as tf
from tkinter import filedialog
from tkinter import *
import logging

# Initialize Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', filename='fvs.log', filemode='w')

# Step 1: Data Loading and Preprocessing
def load_data(data_dir):
    images = []
    labels = []
    for label_dir in os.listdir(data_dir):
        label_path = os.path.join(data_dir, label_dir)
        for img_name in os.listdir(label_path):
            img_path = os.path.join(label_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            images.append(img)
            labels.append(label_dir)
    return np.array(images), np.array(labels)

data_dir = '/path/to/dataset'
images, labels = load_data(data_dir)

# Advanced Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2],
    contrast_stretching=True,
    histogram_equalization=True,
    adaptive_histogram_equalization=True,
    random_erasing=True
)

# Step 2: Model Selection and Evaluation with FaceNet
def build_model():
    base_model = tf.keras.applications.InceptionResNetV2(include_top=False, input_shape=(224, 224, 3), pooling='avg')
    for layer in base_model.layers:
        layer.trainable = False

    x = base_model.output
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=x)
    return model

model = build_model()
model.compile(optimizer=Adam(lr=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# Step 3: Model Training and Optimization
train_images, val_images, train_labels, val_labels = train_test_split(images, labels, test_size=0.2, random_state=42)

train_generator = datagen.flow(train_images, train_labels, batch_size=32)
val_generator = datagen.flow(val_images, val_labels, batch_size=32)

history = model.fit(
    train_generator,
    steps_per_epoch=len(train_images) // 32,
    validation_data=val_generator,
    validation_steps=len(val_images) // 32,
    epochs=25
)

# Step 4: Evaluating the Model
def plot_history(history):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

plot_history(history)

# Step 5: Saving the Model
model.save('face_verification_model.h5')

# Step 6: Inference Function for Face Verification
def verify_face(model, img_path, threshold=0.5):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    prediction = model.predict(img_array)[0][0]
    logging.info(f'Face verification score: {prediction}')
    return prediction > threshold

# Step 7: Liveness Detection with Deep Learning
def detect_liveness(img_path):
    # Placeholder for liveness detection implementation using a pre-trained model
    # This can include techniques like blinking detection, texture analysis, etc.
    return True  # Assuming liveness detected for this example

# Step 8: Ensuring Privacy (Sample Encryption of Image Data)
from cryptography.fernet import Fernet

def encrypt_image(img_path, key):
    with open(img_path, 'rb') as file:
        image_data = file.read()
    fernet = Fernet(key)
    encrypted = fernet.encrypt(image_data)
    return encrypted

def decrypt_image(encrypted_data, key):
    fernet = Fernet(key)
    decrypted = fernet.decrypt(encrypted_data)
    return decrypted

key = Fernet.generate_key()
encrypted_image = encrypt_image('/path/to/test/image.jpg', key)
decrypted_image = decrypt_image(encrypted_image, key)
logging.info('Image Encryption and Decryption Successful')

# Step 9: Implement Biometric Reading for Access (Sample Implementation)
def biometric_reading_for_access(model, img_path):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Access Granted")
            print("Access Granted")
        else:
            logging.warning("Spoof Attempt Detected, Access Denied")
            print("Spoof Attempt Detected, Access Denied")
    else:
        logging.warning("Face Not Verified, Access Denied")
        print("Face Not Verified, Access Denied")

# Step 10: Digital Payments Security (Sample Implementation)
def secure_digital_payment(model, img_path):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Payment Authorized")
            print("Payment Authorized")
        else:
            logging.warning("Spoof Attempt Detected, Payment Denied")
            print("Spoof Attempt Detected, Payment Denied")
    else:
        logging.warning("Face Not Verified, Payment Denied")
        print("Face Not Verified, Payment Denied")

# Step 11: Attendance Tracking (Sample Implementation)
def track_attendance(model, img_path, attendance_db):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Attendance Marked")
            print("Attendance Marked")
            attendance_db.append(img_path)
        else:
            logging.warning("Spoof Attempt Detected, Attendance Not Marked")
            print("Spoof Attempt Detected, Attendance Not Marked")
    else:
        logging.warning("Face Not Verified, Attendance Not Marked")
        print("Face Not Verified, Attendance Not Marked")

attendance_db = []
track_attendance(model, '/path/to/test/image.jpg', attendance_db)
print(attendance_db)

# Step 12: Employee Time and Attendance Tracking (Sample Implementation)
def track_employee_time_and_attendance(model, img_path, employee_db):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Employee Check-In Marked")
            print("Employee Check-In Marked")
            employee_db.append(img_path)
        else:
            logging.warning("Spoof Attempt Detected, Check-In Not Marked")
            print("Spoof Attempt Detected, Check-In Not Marked")
    else:
        logging.warning("Face Not Verified, Check-In Not Marked")
        print("Face Not Verified, Check-In Not Marked")

employee_db = []
track_employee_time_and_attendance(model, '/path/to/test/image.jpg', employee_db)
print(employee_db)

# Step 13: Simple GUI using Tkinter
def browse_image():
    filename = filedialog.askopenfilename()
    return filename

def verify_image():
    img_path = browse_image()
    is_verified = verify_face(model, img_path)
    if is_verified:
        print("Face Verified")
    else:
        print("Face Not Verified")

root = Tk()
root.title("Face Verification System")
root.geometry("300x200")

btn_verify = Button(root, text="Verify Face", command=verify_image)
btn_verify.pack(pady=20)

root.mainloop()

# Finished Implementation


In [ ]:
# Importing Required Libraries
import numpy as np
import pandas as pd
import cv2
import os
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import tensorflow as tf
from tkinter import filedialog
from tkinter import *
import logging

# Initialize Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', filename='fvs.log', filemode='w')

# Step 1: Data Loading and Preprocessing
def load_data(data_dir):
    images = []
    labels = []
    for label_dir in os.listdir(data_dir):
        label_path = os.path.join(data_dir, label_dir)
        for img_name in os.listdir(label_path):
            img_path = os.path.join(label_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            images.append(img)
            labels.append(label_dir)
    return np.array(images), np.array(labels)

data_dir = '/path/to/dataset'
images, labels = load_data(data_dir)

# Advanced Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2],
    contrast_stretching=True,
    histogram_equalization=True,
    adaptive_histogram_equalization=True,
    random_erasing=True
)

# Step 2: Model Selection and Evaluation with FaceNet
def build_model():
    base_model = tf.keras.applications.InceptionResNetV2(include_top=False, input_shape=(224, 224, 3), pooling='avg')
    for layer in base_model.layers:
        layer.trainable = False

    x = base_model.output
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=x)
    return model

model = build_model()
model.compile(optimizer=Adam(lr=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# Step 3: Model Training and Optimization
train_images, val_images, train_labels, val_labels = train_test_split(images, labels, test_size=0.2, random_state=42)

train_generator = datagen.flow(train_images, train_labels, batch_size=32)
val_generator = datagen.flow(val_images, val_labels, batch_size=32)

history = model.fit(
    train_generator,
    steps_per_epoch=len(train_images) // 32,
    validation_data=val_generator,
    validation_steps=len(val_images) // 32,
    epochs=25
)

# Step 4: Evaluating the Model
def plot_history(history):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

plot_history(history)

# Step 5: Saving the Model
model.save('face_verification_model.h5')

# Step 6: Inference Function for Face Verification
def verify_face(model, img_path, threshold=0.5):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    prediction = model.predict(img_array)[0][0]
    logging.info(f'Face verification score: {prediction}')
    return prediction > threshold

# Step 7: Liveness Detection with Deep Learning
def detect_liveness(img_path):
    # Placeholder for liveness detection implementation using a pre-trained model
    # This can include techniques like blinking detection, texture analysis, etc.
    return True  # Assuming liveness detected for this example

# Step 8: Ensuring Privacy (Sample Encryption of Image Data)
from cryptography.fernet import Fernet

def encrypt_image(img_path, key):
    with open(img_path, 'rb') as file:
        image_data = file.read()
    fernet = Fernet(key)
    encrypted = fernet.encrypt(image_data)
    return encrypted

def decrypt_image(encrypted_data, key):
    fernet = Fernet(key)
    decrypted = fernet.decrypt(encrypted_data)
    return decrypted

key = Fernet.generate_key()
encrypted_image = encrypt_image('/path/to/test/image.jpg', key)
decrypted_image = decrypt_image(encrypted_image, key)
logging.info('Image Encryption and Decryption Successful')

# Step 9: Implement Biometric Reading for Access (Sample Implementation)
def biometric_reading_for_access(model, img_path):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Access Granted")
            print("Access Granted")
        else:
            logging.warning("Spoof Attempt Detected, Access Denied")
            print("Spoof Attempt Detected, Access Denied")
    else:
        logging.warning("Face Not Verified, Access Denied")
        print("Face Not Verified, Access Denied")

# Step 10: Digital Payments Security (Sample Implementation)
def secure_digital_payment(model, img_path):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Payment Authorized")
            print("Payment Authorized")
        else:
            logging.warning("Spoof Attempt Detected, Payment Denied")
            print("Spoof Attempt Detected, Payment Denied")
    else:
        logging.warning("Face Not Verified, Payment Denied")
        print("Face Not Verified, Payment Denied")

# Step 11: Attendance Tracking (Sample Implementation)
def track_attendance(model, img_path, attendance_db):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Attendance Marked")
            print("Attendance Marked")
            attendance_db.append(img_path)
        else:
            logging.warning("Spoof Attempt Detected, Attendance Not Marked")
            print("Spoof Attempt Detected, Attendance Not Marked")
    else:
        logging.warning("Face Not Verified, Attendance Not Marked")
        print("Face Not Verified, Attendance Not Marked")

attendance_db = []
track_attendance(model, '/path/to/test/image.jpg', attendance_db)
print(attendance_db)

# Step 12: Employee Time and Attendance Tracking (Sample Implementation)
def track_employee_time_and_attendance(model, img_path, employee_db):
    is_verified = verify_face(model, img_path)
    if is_verified:
        liveness = detect_liveness(img_path)
        if liveness:
            logging.info("Employee Check-In Marked")
            print("Employee Check-In Marked")
            employee_db.append(img_path)
        else:
            logging.warning("Spoof Attempt Detected, Check-In Not Marked")
            print("Spoof Attempt Detected, Check-In Not Marked")
    else:
        logging.warning("Face Not Verified, Check-In Not Marked")
        print("Face Not Verified, Check-In Not Marked")

employee_db = []
track_employee_time_and_attendance(model, '/path/to/test/image.jpg', employee_db)
print(employee_db)

# Step 13: Simple GUI using Tkinter
def browse_image():
    filename = filedialog.askopenfilename()
    return filename

def verify_image():
    img_path = browse_image()
    is_verified = verify_face(model, img_path)
    if is_verified:
        print("Face Verified")
    else:
        print("Face Not Verified")

root = Tk()
root.title("Face Verification System")
root.geometry("300x200")

btn_verify = Button(root, text="Verify Face", command=verify_image)
btn_verify.pack(pady=20)

root.mainloop()

# Finished Implementation
